<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/baobab_bolt_food_groceries_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


After running the cell above and following the authentication steps, your Google Drive will be mounted at `/content/drive`. You can then access your files using paths like `/content/drive/My Drive/path/to/your_data.csv`.

In [5]:
import pandas as pd
import io # Not strictly needed for pd.read_csv directly from URL, but harmless.
import re # Needed for robust URL parsing

# Placeholder for your Google Sheet shareable link.
# Make sure the sharing settings are set to 'Anyone with the link can view'.
# Example: 'https://docs.google.com/spreadsheets/d/1B_Asx8b_G_c38x1_i_r2B8g_F1_y_S_d/edit#gid=0'
google_sheet_url = 'https://docs.google.com/spreadsheets/d/1k4toaIWRj9oUDoXvdqsZQhLl0TgPRCHDY1doJPWjAEA/edit?gid=493188322#gid=493188322'

try:
    # Use regex to extract the document ID and gid from the Google Sheet URL
    # This pattern is more robust for various Google Sheet URL formats
    match = re.search(r'/d/([a-zA-Z0-9_-]+)(?:/edit)?(?:.*[?&]gid=([0-9]+))?(?:.*#gid=([0-9]+))?', google_sheet_url)

    if not match:
        raise ValueError("Could not parse Google Sheet ID or GID from the URL. Please check the format.")

    sheet_id = match.group(1)
    # The gid can be in either query parameter (?gid=) or fragment (#gid=)
    # Prioritize the gid from the query parameter, then fragment, default to '0'
    gid = match.group(2) if match.group(2) else (match.group(3) if match.group(3) else '0')

    # Construct the export URL
    export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

    df = pd.read_csv(export_url)
    print("Google Sheet loaded successfully!")
    print(df.head())
except Exception as e:
    print(f"An error occurred while loading the Google Sheet: {e}")
    print("Please ensure the Google Sheet link is correct and publicly accessible (or shared with 'Anyone with the link').")
    print("You might need to adjust the sharing settings of your Google Sheet to 'Anyone with the link can view'.")

An error occurred while loading the Google Sheet: HTTP Error 401: Unauthorized
Please ensure the Google Sheet link is correct and publicly accessible (or shared with 'Anyone with the link').
You might need to adjust the sharing settings of your Google Sheet to 'Anyone with the link can view'.


In [6]:
# Install necessary libraries
!pip -q install gspread gspread-dataframe

In [7]:
from google.colab import auth
auth.authenticate_user()

In [9]:
import gspread
from google.auth import default
from gspread_dataframe import get_as_dataframe
import pandas as pd

# Your Google Sheet URL (without the /edit?gid=... part, gspread handles the sheet ID)
SSOT_GSHEET_URL = "https://docs.google.com/spreadsheets/d/1k4toaIWRj9oUDoXvdqsZQhLl0TgPRCHDY1doJPWjAEA"

# The name of the specific worksheet (tab) you want to load
SOURCE_TAB = "raw_bolt_food_email"

creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_url(SSOT_GSHEET_URL)

# Load data from the specified worksheet into a DataFrame
df = get_as_dataframe(sh.worksheet(SOURCE_TAB), header=0, dtype=str)

# Clean up the DataFrame as per your example
df = df.dropna(how="all").copy()
df.columns = [c.lower() for c in df.columns]

print("Google Sheet loaded successfully using gspread!")
print(df.head())

Google Sheet loaded successfully using gspread!
         source_id      source_name  source_type           email_date  \
0  bolt_food_email  Bolt Food Email  app_receipt  29/04/2026 22:00:20   
1  bolt_food_email  Bolt Food Email  app_receipt  28/04/2026 15:26:17   
2  bolt_food_email  Bolt Food Email  app_receipt  27/04/2026 13:14:05   
3  bolt_food_email  Bolt Food Email  app_receipt  25/04/2026 21:09:46   
4  bolt_food_email  Bolt Food Email  app_receipt  22/04/2026 13:14:53   

                     sender                  subject  gmail_message_id  \
0  Bolt Food <info@bolt.eu>  Delivery from Bolt Food  19ddb41e039d2166   
1  Bolt Food <info@bolt.eu>  Delivery from Bolt Food  19dd4b2c015ed90e   
2  Bolt Food <info@bolt.eu>  Delivery from Bolt Food  19dcf135e3d1ce13   
3  Bolt Food <info@bolt.eu>  Delivery from Bolt Food  19dc67a27d57f420   
4  Bolt Food <info@bolt.eu>  Delivery from Bolt Food  19db5544ec21df97   

          thread_id                                           raw_bo

In [10]:
print('DataFrame Info:')
df.info()

print('\nFirst 5 rows of the raw_body column:')
for i, body in enumerate(df['raw_body'].head()):
    print(f'--- Email Body {i+1} ---\n{body}\n--------------------')

print('\nTop 5 unique subjects:')
print(df['subject'].value_counts().head())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 128 entries, 0 to 127
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   source_id         128 non-null    object
 1   source_name       128 non-null    object
 2   source_type       128 non-null    object
 3   email_date        128 non-null    object
 4   sender            128 non-null    object
 5   subject           128 non-null    object
 6   gmail_message_id  128 non-null    object
 7   thread_id         128 non-null    object
 8   raw_body          128 non-null    object
 9   raw_snippet       128 non-null    object
 10  has_attachments   128 non-null    object
 11  ingested_at       128 non-null    object
 12  parse_status      128 non-null    object
dtypes: object(13)
memory usage: 14.0+ KB

First 5 rows of the raw_body column:
--- Email Body 1 ---
 
­2026-04-29 


*Hello, Habib!*

This is your receipt.
From Juls Fresh Fruit & Wagashi ­JPMV

In [11]:
import re

# Convert 'email_date' to datetime objects
df['email_date'] = pd.to_datetime(df['email_date'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Filter out rows where email_date could not be parsed (if any)
df.dropna(subset=['email_date'], inplace=True)

# Function to extract items and quantities from raw_body
def extract_items(raw_body):
    items = []
    # Regex to find lines like 'QUANTITY × ITEM_NAME'
    # It looks for a number, followed by ' × ', then captures the rest of the line until a currency symbol or new line
    # Adjusted regex to handle potential GH₵ amount after item name or on next line if it's the item price
    item_pattern = re.compile(r'^(\d+)\s×\s(.*?)(?:\sGH\u20b5|\n|$)', re.MULTILINE)

    # Split the body into lines to process each potential item line
    lines = raw_body.split('\n')
    for line in lines:
        match = item_pattern.match(line.strip())
        if match:
            quantity = int(match.group(1))
            item_name = match.group(2).strip()
            # Further clean item_name if it contains trailing currency amounts incorrectly matched
            item_name = re.sub(r'GH\u20b5.*$', '', item_name).strip()
            items.append({'item': item_name, 'quantity': quantity})
    return items

# Apply the function to the DataFrame and create a list of all extracted items
all_items = []
for index, row in df.iterrows():
    extracted_items = extract_items(row['raw_body'])
    for item in extracted_items:
        all_items.append({
            'email_date': row['email_date'],
            'item': item['item'],
            'quantity': item['quantity'],
            'gmail_message_id': row['gmail_message_id'] # Keep original email identifier
        })

# Create a new DataFrame from the extracted items
items_df = pd.DataFrame(all_items)

print("Extracted Items DataFrame Info:")
items_df.info()
print("\nFirst 5 rows of the Extracted Items DataFrame:")
print(items_df.head())

Extracted Items DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   email_date        263 non-null    datetime64[ns]
 1   item              263 non-null    object        
 2   quantity          263 non-null    int64         
 3   gmail_message_id  263 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 8.3+ KB

First 5 rows of the Extracted Items DataFrame:
           email_date                                 item  quantity  \
0 2026-04-29 22:00:20  WAGASHI (Medium) Khebab/Suya powder         1   
1 2026-04-28 15:26:17                         Fish fillets         1   
2 2026-04-27 13:14:05             Ahmad Lemon & Ginger 40G         1   
3 2026-04-27 13:14:05    Ramli Uht Milk Full Cream 3.5% 1L         1   
4 2026-04-27 13:14:05                       Coca Cola 1.5L     

In [12]:
# Sort the DataFrame by item and then by email_date
items_df_sorted = items_df.sort_values(by=['item', 'email_date']).copy()

# Calculate the time difference between consecutive purchases for each item
# This gives us the replenishment time
items_df_sorted['replenishment_time'] = items_df_sorted.groupby('item')['email_date'].diff()

# Filter out the first purchase for each item, as it won't have a replenishment time
replenishment_df = items_df_sorted.dropna(subset=['replenishment_time']).copy()

print("Replenishment DataFrame Info:")
replenishment_df.info()
print("\nFirst 5 rows of the Replenishment DataFrame (showing replenishment times):")
print(replenishment_df.head())

Replenishment DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 147 entries, 168 to 229
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   email_date          147 non-null    datetime64[ns] 
 1   item                147 non-null    object         
 2   quantity            147 non-null    int64          
 3   gmail_message_id    147 non-null    object         
 4   replenishment_time  147 non-null    timedelta64[ns]
dtypes: datetime64[ns](1), int64(1), object(2), timedelta64[ns](1)
memory usage: 6.9+ KB

First 5 rows of the Replenishment DataFrame (showing replenishment times):
             email_date                      item  quantity  gmail_message_id  \
168 2025-12-26 20:00:36  Ahmad Lemon & Ginger 40G         1  19b5c3fb13ee8a17   
77  2026-03-18 12:05:16  Ahmad Lemon & Ginger 40G         1  19d00d5fc21c0c97   
64  2026-03-26 18:29:46  Ahmad Lemon & Ginger 40G         1  1

In [ ]:
# Calculate descriptive statistics for replenishment_time for each item
replenishment_summary = replenishment_df.groupby('item')['replenishment_time'].agg(
    ['mean', 'median', 'std', 'count']
)

# Convert timedelta to more readable units (e.g., days)
replenishment_summary['mean_days'] = replenishment_summary['mean'].dt.days
replenishment_summary['median_days'] = replenishment_summary['median'].dt.days
replenishment_summary['std_days'] = replenishment_summary['std'].dt.days

# Drop the original timedelta columns if you prefer to only see days
replenishment_summary = replenishment_summary.drop(columns=['mean', 'median', 'std'])

# Sort by mean replenishment time to see the fastest/slowest replenished items
replenishment_summary = replenishment_summary.sort_values(by='mean_days').reset_index()

print("Replenishment Summary by Item (in days):")
print(replenishment_summary.head(10))